In [5]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Load data
train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')

In [6]:
def calculate_aas(df):
    aas = np.zeros(len(df))
    aas += np.where(df['stroke_density'] > 0.7, 2, 0)
    aas += np.where(df['complexity'] > 0.65, 2, 0)
    aas += np.where(df['uses_gold_leaf'] == True, 1, 0)
    aas += np.where(df['has_signature'] == True, 1, 0)
    aas += np.where((df['num_colors'] > 65) & (df['colorfulness'] > 0.7), 2, 0)
    aas -= np.where((df['contrast'] < 0.4) | (df['brightness'] < 0.45) | (df['brightness'] > 0.75), 1, 0)
    return np.where(aas >= 5, 'Autentic', 'Incert')

test_task1_labels = calculate_aas(test)

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

features_for_clustering = [
    'painter_style_score', 'fake_style_score', 'stroke_density', 
    'complexity', 'colorfulness', 'brightness', 'contrast', 'num_colors'
]

scaler = StandardScaler()
X_cluster_all = scaler.fit_transform(pd.concat([train[features_for_clustering], test[features_for_clustering]]))

kmeans = KMeans(n_clusters=5, random_state=42, n_init=20, max_iter = 500)
kmeans.fit(X_cluster_all)
test_task2_labels = kmeans.predict(scaler.transform(test[features_for_clustering]))

In [13]:
import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.linear_model import Ridge
from sklearn.preprocessing import OrdinalEncoder, StandardScaler
from sklearn.cluster import KMeans
from sklearn.model_selection import KFold
from sklearn.metrics import mean_absolute_error

# 1. Load Data
train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')

# 2. Data Cleaning & Feature Engineering
def preprocess_data(df):
    df = df.copy()
    # Extract canvas dimensions and physical area
    dims = df['canvas_size'].str.split('x', expand=True).astype(float)
    df['canvas_width'] = dims[0]
    df['canvas_height'] = dims[1]
    df['canvas_area'] = df['canvas_width'] * df['canvas_height']
    df['canvas_aspect_ratio'] = df['canvas_width'] / df['canvas_height']
    return df

train_df = preprocess_data(train)
test_df = preprocess_data(test)

# Define feature columns
drop_cols = ['SampleID', 'canvas_size', 'target_price']
features = [c for c in train_df.columns if c not in drop_cols]
cat_features = ['brush_type', 'dominant_color', 'art_period_hint', 'auction_house', 'image_quality']

X_train = train_df[features].copy()
y_train = train_df['target_price']
X_test = test_df[features].copy()

# One-Hot Encoding for Linear/Ridge Regressor
X_train_ohe = pd.get_dummies(X_train, columns=cat_features, drop_first=True)
X_test_ohe = pd.get_dummies(X_test, columns=cat_features, drop_first=True)
X_train_ohe, X_test_ohe = X_train_ohe.align(X_test_ohe, join='left', axis=1, fill_value=0)

scaler = StandardScaler()
X_tr_scaled = scaler.fit_transform(X_train_ohe)
X_te_scaled = scaler.transform(X_test_ohe)

# 3. Train Regressor with 5-Fold Cross Validation
kf = KFold(n_splits=5, shuffle=True, random_state=42)
oof_preds = np.zeros(len(train_df))
test_preds = np.zeros(len(test_df))

for fold, (tr, va) in enumerate(kf.split(X_tr_scaled, y_train)):
    regressor = Ridge(alpha=10.0)
    regressor.fit(X_tr_scaled[tr], y_train.iloc[tr])
    
    oof_preds[va] = regressor.predict(X_tr_scaled[va])
    test_preds += regressor.predict(X_te_scaled) / kf.n_splits

cv_mae = mean_absolute_error(y_train, oof_preds)
print(f"5-Fold CV MAE: {cv_mae:.2f}")

# 4. Generate Subtask Predictions
# Subtask 1: AAS Rule Calculation
def compute_task1(df):
    aas = np.zeros(len(df), dtype=int)
    aas += np.where(df['stroke_density'] > 0.7, 2, 0)
    aas += np.where(df['complexity'] > 0.65, 2, 0)
    aas += np.where(df['uses_gold_leaf'] == True, 1, 0)
    aas += np.where(df['has_signature'] == True, 1, 0)
    aas += np.where((df['num_colors'] > 65) & (df['colorfulness'] > 0.7), 2, 0)
    cond_sub = (df['contrast'] < 0.4) | (df['brightness'] < 0.45) | (df['brightness'] > 0.75)
    aas -= np.where(cond_sub, 1, 0)
    return np.where(aas >= 5, 'Autentic', 'Incert')

task1_preds = compute_task1(test)

# Subtask 2: Painter Grouping (k=5 Clustering)
cluster_cols = ['painter_style_score', 'fake_style_score', 'stroke_density', 'complexity', 'colorfulness', 'brightness', 'contrast', 'num_colors']
X_cluster = scaler.fit_transform(pd.concat([train[cluster_cols], test[cluster_cols]], axis=0))
kmeans = KMeans(n_clusters=5, random_state=42, n_init=20).fit(X_cluster)
task2_preds = kmeans.predict(scaler.transform(test[cluster_cols]))

# Subtask 3: Predicted Prices (Rounded Integers)
task3_preds = np.round(test_preds).astype(int)

# 5. Format & Save Submission File
sub_t1 = pd.DataFrame({'SampleID': test['SampleID'], 'subtaskID': 'Task1', 'Answer': task1_preds})
sub_t2 = pd.DataFrame({'SampleID': test['SampleID'], 'subtaskID': 'Task2', 'Answer': task2_preds})
sub_t3 = pd.DataFrame({'SampleID': test['SampleID'], 'subtaskID': 'Task3', 'Answer': task3_preds})

submission = pd.concat([sub_t1, sub_t2, sub_t3], ignore_index=True)
submission.to_csv('submission.csv', index=False)
print("submission.csv generated successfully!")

5-Fold CV MAE: 717.16
submission.csv generated successfully!


In [12]:
sub_t1 = pd.DataFrame({
    'SampleID': test['SampleID'],
    'subtaskID': 'Task1',
    'Answer': test_task1_labels
})

sub_t2 = pd.DataFrame({
    'SampleID': test['SampleID'],
    'subtaskID': 'Task2',
    'Answer': test_task2_labels
})

sub_t3 = pd.DataFrame({
    'SampleID': test['SampleID'],
    'subtaskID': 'Task3',
    'Answer': 0
})

submission = pd.concat([sub_t1, sub_t2, sub_t3], ignore_index=True)
submission.to_csv('submission.csv', index=False)
print("submission.csv generated successfully!")

submission.csv generated successfully!
